# Curso: Redes Neurais Convolucionais

Prof. [Denilson Alves Pereira](https://sites.google.com/ufla.br/denilsonpereira) <br>
Departamento de Ciência da Computação (DCC) <br>
Instituto de Ciências Exatas e Tecnológicas (ICET) <br> 
Universidade Federal de Lavras (UFLA)

# Atividade Prática 03

**Instruções:**
1. Siga os passos indicados em cada célula abaixo para completar a atividade.
2. Você deve inserir código somente entre as linhas marcadas com **INICIE O CÓDIGO AQUI** e **TERMINE O CÓDIGO AQUI**. Há uma indicação de quantas linhas de código são necessárias.
3. Em alguns pontos, confira o resultado esperado conforme marcado com **SAÍDA ESPERADA**.

**Tempo estimado para execução**: 0,5 hora

Versão: Setembro, 2021

## O Problema a ser Resolvido

O objetivo da atividade é elaborar uma aplicação de rede neural convolucional para detecção de objetos em imagens. <br>

Você vai praticar as seguintes habilidades:
- Aprender a usar um modelo pré-treinado para detecção de objetos baseado no algoritmo YOLO
- Implementar uma função para reconhecimento, localização e classificação de diversos tipos de objetos em imagens

### YOLO

Para a detecção de objetos, será usada uma implementação do algoritmo denominado YOLO (*You Only Look Once*), popular na área de visão computacional. Ele fornece uma série de modelos de deep learning para detecção rápida de objetos.

A abordagem do YOLO envolve uma rede neural convolucional, originalmente uma versão da GoogLeNet, que depois foi atualizada e chamada de DarkNet baseada na VGG. A estratégia divide a imagem de entrada em uma grade de células, e cada célula classifica objetos contidos em caixas delimitadoras (*bounding box*). O resultado é um grande número de caixas delimitadores candidatas que são consolidadas em uma predição final por meio de um passo de pós-processamento. Há algumas variações da abordagem (YOLOv1, YOLOv2, YOLOv3), que correspondem a refinamentos de melhoria no projeto.

Exemplo adaptado de https://www.analyticsvidhya.com/blog/2021/06/yolo-object-detection-in-seconds-cvlib/?utm_source=feed&utm_medium=feed-articles&utm_campaign=feed

Outros exemplos: <br>
https://www.kaggle.com/zikazika/what-is-object-detection-yolo <br>
https://www.kaggle.com/ashrafkhan94/object-detection-with-yolov3 <br>
https://www.kaggle.com/balraj98/yolo-v5-car-object-detection <br>
https://www.kaggle.com/uygarusta/yolov3-opencv-car-detection <br>
https://www.kaggle.com/ashishsingh226/car-detection-using-maskrcnn

## Pacotes

Usaremos neste exemplo uma implementação do algoritmo YOLO disponível na biblioteca ['cvlib'](https://www.cvlib.net/).

A biblioteca cvlib pode ser instalada com 'pip install cvlib'. <br>
cvlib depende de 'OpenCV' (pip install OpenCV-python) e 'Tensorflow' (pip install TensorFlow), normalmente já incluídos no pacote Anaconda.

In [ ]:
from IPython.display import Image, display # command shell for interactive computing
import os # operating system library
import cv2 # OpenCV library for computer vision problems
import cvlib as cv # computer vision library for object detection
from cvlib.object_detection import draw_bbox # draw bounding boxes

## Inicializações

In [ ]:
DIR_DATASET = './datasets/object-detection/'

## Leitura das Imagens

A pasta (diretório) do dataset contém imagens de carros. Escolha algumas delas e adicione o nome de seus arquivos no código abaixo para carregá-las. <br>
Inclua também algumas imagens da pasta 'cats-and-dogs', preferencialmente imagens de animais que não ocupem toda a imagem (um gato junto com uma pessoa, por exemplo). <br>
Além disso, adicione outras imagens de sua escolha que contenham objetos diversos, tais como pessoas, bicicletas, ônibus, caminhões e frutas.

In [ ]:
# Algumas imagens de exemplo
### INICIE O CÓDIGO AQUI ### (2 ou 3 linhas)

### TERMINE O CÓDIGO AQUI ###
for image_file in image_files:
    print('\n\nArquivo:', image_file)
    display(Image(filename=DIR_DATASET+image_file))

## Detecção de Objetos

A detecção de objetos será feita pela biblioteca 'cvlib', por meio da função 'cvlib.detect_common_objects'. Essa função recebe como entrada uma imagem formatada como um array NumPy e retorna as três seguintes listas:
- bbox: lista de coordenadas das fronteiras dos objetos detectados, na forma de um retângulo.
- label: lista de rótulos dos objetos detectados na mesma ordem da lista 'bbox'.
- conf: lista de pontuações de confiança para as detecções, as quais representam a confiança do modelo de que o resultado predito está correto.

A biblioteca 'cvlib' disponibiliza vários modelos pré-treinados pelo algoritmo YOLO. O modelo padrão é o 'yolov3'. Na função abaixo, nós vamos usar o modelo 'yolov3-tiny', uma versão simplificada que executa mais rápido e usa menos memória ao custo de uma redução no desempenho do modelo.

In [ ]:
def detect_box(filename, model="yolov3-tiny", confidence=0.6):
    """
    Detects common objects on an image and return their bounding boxes.
    
    Arguments:
        filename (str): Filename of the image.
        model (str): Either "yolov3" or "yolov3-tiny". Defaults to "yolov3-tiny".
        confidence (float, optional): Desired confidence level. Defaults to 0.6.
            This confidence parameter is a threshold parameter for selection only those bounding boxes
            whose confidence is greater than this provided confidence.
    
    Returns:
        bbox: lista de coordenadas das fronteiras dos objetos detectados, na forma de um retângulo.
        label: lista de rótulos dos objetos detectados na mesma ordem da lista 'bbox'.
        conf: lista de pontuações de confiança para as detecções, as quais representam a confiança do modelo de que o resultado predito está correto.
        img: image in a numpy array
    """

    # Read the image into a numpy array
    img = cv2.imread(filename)
    # Perform the object detection
    bbox, label, conf = cv.detect_common_objects(img, confidence=confidence, model=model)
    
    return bbox, label, conf, img

In [ ]:
def save_and_draw_box(filename, bbox, label, conf, img):
    """
    Save and draw image with the bounding boxes of the detected objects.
    
    Arguments:
        filename (str): Filename of the image.
        bbox: lista de coordenadas das fronteiras dos objetos detectados, na forma de um retângulo.
        label: lista de rótulos dos objetos detectados na mesma ordem da lista 'bbox'.
        conf: lista de pontuações de confiança para as detecções, as quais representam a confiança do modelo de que o resultado predito está correto.
        img: image in a numpy array
    """

    # Print detected objects with confidence level
    print('\n\nImage processed: ', filename, '\n')
    for l, c in zip(label, conf):
        print('Detected object: ', l, ' - confidence level = ', c, '\n')
    # Create a new image that includes the bounding boxes
    output_image = draw_bbox(img, bbox, label, conf)
    # Save the image in the directory images-with-boxes
    filename = filename.split('/')[-1]
    cv2.imwrite(DIR_DATASET+'images-with-boxes/'+filename, output_image)
    # Display the image with bounding boxes
    display(Image(DIR_DATASET+'images-with-boxes/'+filename))

Cria um diretório para armazenar as imagens com as marcações em caixa dos objetos

In [ ]:
dir_box_name = DIR_DATASET+'images-with-boxes'
if not os.path.exists(dir_box_name):
    os.mkdir(dir_box_name)

Detecta e desenha a imagem com os objetos detectados

In [ ]:
for image_file in image_files:
    bbox, label, conf, img = detect_box(DIR_DATASET+image_file, model="yolov3-tiny", confidence=0.6)
    save_and_draw_box(DIR_DATASET+image_file, bbox, label, conf, img)

Reduza o valor do nível de confiança da detecção (parâmetro 'confidence'), por exemplo para 0.2, e verifique o resultado. Provavelmente, mais objetos serão detectados, porém podem ocorrer mais erros na detecção.

In [ ]:
### INICIE O CÓDIGO AQUI ### (alteração em uma linha)
for image_file in image_files:
    bbox, label, conf, img = detect_box(DIR_DATASET+image_file, model="yolov3-tiny", confidence=0.6)
    save_and_draw_box(DIR_DATASET+image_file, bbox, label, conf, img)
### TERMINE O CÓDIGO AQUI ###

Troque o modelo de 'yolov3-tiny' para 'yolov3'. Observe que o tempo de processamento deve aumentar, porém pode haver melhoria no resultado.

In [ ]:
### INICIE O CÓDIGO AQUI ### (alteração em uma linha)
for image_file in image_files:
    bbox, label, conf, img = detect_box(DIR_DATASET+image_file, model="yolov3-tiny", confidence=0.6)
    save_and_draw_box(DIR_DATASET+image_file, bbox, label, conf, img)
### TERMINE O CÓDIGO AQUI ###

# Fim

Parabéns! Você efetuou todos os passos para elaborar uma aplicação de rede neural convolucional para detecção de objetos em imagens.

-------------------------------------------